# A Program Is Running but You Cannot Find Its PID

This lab simulates a realistic production incident relevant to an **AI/ML Engineer** running services such as:

- FastAPI inference servers
- Python training jobs
- RAG pipelines
- Model-serving processes
- Kafka consumers
- Celery workers
- GPU inference workloads

---

## 1. Objective

### Scenario

A service appears to be running:

```text
CPU usage is increasing
Memory is being consumed
A port is responding
Logs are being generated
```

But when you try:

In [ ]:
!ps aux | grep my_program

You cannot find the expected process or PID.

Your job is to:

```text
1. Identify system requirements
2. Establish a baseline
3. Simulate the problem
4. Investigate the running process
5. Find the PID
6. Analyze why it was difficult to find
7. Resolve the problem
8. Confirm resolution
9. Measure impact
10. Connect it to AI/ML production use cases
```

---

# Phase 1: Identify System Requirements

## Check Linux Distribution

In [ ]:
!cat /etc/os-release

Expected information:

```text
NAME="Ubuntu"
VERSION="..."
```

---

## Check Kernel

In [ ]:
!uname -r

---

## Check CPU

In [ ]:
!lscpu

Important information:

```text
Architecture
CPU(s)
Model name
```

---

## Check Memory

In [ ]:
!free -h

Example:

```text
               total        used        free
Mem:           16Gi        5Gi         8Gi
```

---

## Check Disk Space

In [ ]:
!df -h

Focus on:

```text
/
```

---

## Check Available Troubleshooting Tools

In [ ]:
!which ps
!which top
!which htop
!which pgrep
!which lsof
!which ss
!which systemctl

Install useful tools if necessary:

In [ ]:
!apt update
!apt install -y htop lsof procps

### Success Criteria

You should have:

```text
ps
top/htop
pgrep
lsof
ss
systemctl
```

---

# Phase 2: Establish System Baseline

Before creating the problem, inspect the current system.

## Check Running Processes

In [ ]:
!ps aux --sort=-%mem | head

Check CPU-heavy processes:

In [ ]:
!ps aux --sort=-%cpu | head

---

## Check Total Process Count

In [ ]:
!ps -e | wc -l

---

## Check Memory

In [ ]:
!free -h

---

## Check Load

In [ ]:
!uptime

Record these values.

### Baseline Example

```text
Memory Used: 4 GB
CPU Load: 0.30
Processes: 220
```

This is important because production troubleshooting requires comparing:

```text
Before Incident
       ↓
Incident
       ↓
After Resolution
```

---

# Phase 3: Simulate the Problem

We will create a Python process with a misleading process name.

Create:

In [ ]:
!nano hidden_worker.py

Add:

In [ ]:
import time
import os

print(f"Worker started with PID: {os.getpid()}")

data = []

while True:
    data.append("X" * 1024 * 1024)
    print(f"PID={os.getpid()} | Memory blocks={len(data)}")
    time.sleep(5)

Run it in the background:

In [ ]:
!python3 hidden_worker.py &

Check the shell PID:

In [ ]:
!echo $!

Suppose:

```text
12345
```

Now imagine that you forgot the PID.

---

# Phase 4: The Problem

You know something is consuming memory.

Check:

In [ ]:
!free -h

Then:

In [ ]:
!top

or:

In [ ]:
!htop

You may see:

```text
PID      COMMAND      %CPU     %MEM
12345    python3      2.0      10.5
```

But imagine you search for:

In [ ]:
!ps aux | grep hidden_worker

And you don't find what you expect.

This commonly happens because:

```text
You search for the wrong process name
        ↓
The shell script has exited
        ↓
The real process is a child process
        ↓
The application changed its process name
        ↓
The process is running inside Docker
        ↓
The process is running under systemd
        ↓
The process belongs to another user
        ↓
The process runs inside another namespace
```

---

# Phase 5: Investigation Workflow

## Step 1: Search by Process Name

In [ ]:
!pgrep -af python

The `-a` shows the command line.

Example:

```text
12345 python3 hidden_worker.py
```

Now you have the PID.

---

## Step 2: Search All Python Processes

In [ ]:
!ps -ef | grep python

Better:

In [ ]:
!pgrep -af python3

---

## Step 3: Sort by Memory

If you don't know the application name:

In [ ]:
!ps aux --sort=-%mem | head -20

Example:

```text
USER     PID   %MEM COMMAND
user     12345 12.3 python3 hidden_worker.py
```

This is extremely useful for AI/ML workloads.

---

## Step 4: Sort by CPU

In [ ]:
!ps aux --sort=-%cpu | head -20

Useful when:

```text
Inference latency increases
CPU suddenly reaches 100%
A training job becomes stuck
Embedding generation becomes slow
```

---

# Phase 6: Investigate `/proc`

Once you find the PID:

In [ ]:
!PID=12345

Check:

In [ ]:
!ls /proc/$PID

---

## Full Command

In [ ]:
!tr '\0' ' ' < /proc/$PID/cmdline

Example:

```text
python3 hidden_worker.py
```

---

## Current Working Directory

In [ ]:
!readlink /proc/$PID/cwd

---

## Executable

In [ ]:
!readlink /proc/$PID/exe

---

## Environment

In [ ]:
!tr '\0' '\n' < /proc/$PID/environ

⚠️ Be careful: environment variables may contain secrets.

---

# Phase 7: Check Process Tree

A major reason you cannot find a PID is that the process is actually a child process.

Run:

In [ ]:
!pstree -p

Example:

```text
systemd(1)
 └── python3(12345)
```

Or:

```text
gunicorn(1000)
 ├── python(1001)
 ├── python(1002)
 └── python(1003)
```

This is extremely common with AI APIs.

For example:

```text
systemd
   │
   └── gunicorn
          │
          ├── FastAPI Worker
          ├── FastAPI Worker
          └── FastAPI Worker
```

You may search for:

```text
FastAPI
```

But Linux only shows:

```text
gunicorn
python
```

---

# Phase 8: Check Network Ports

Suppose your AI API is responding on:

```text
Port 8000
```

Find the PID:

In [ ]:
!sudo lsof -i :8000

Or:

In [ ]:
!sudo ss -ltnp | grep :8000

Example:

```text
LISTEN 0 4096 0.0.0.0:8000
users:(("python3",pid=12345))
```

Now you know:

```text
Port 8000
    ↓
PID 12345
    ↓
Python Process
    ↓
Application Command
```

---

# Phase 9: Check systemd

Your application may not be visible the way you expect because systemd started it.

Search:

In [ ]:
!systemctl list-units --type=service

Suppose:

```text
ai-inference.service
```

Check:

In [ ]:
!systemctl status ai-inference.service

You may see:

```text
Main PID: 12345
```

Check logs:

In [ ]:
!journalctl -u ai-inference.service -n 50

---

# Phase 10: Docker Scenario

A very common AI/ML production issue:

```text
Host Linux
    │
    └── Docker Container
            │
            └── Python Model Server
```

Check containers:

In [ ]:
!docker ps

Example:

```text
CONTAINER ID   IMAGE              NAMES
abc123         ai-model-server    inference-api
```

Check processes:

In [ ]:
!docker top inference-api

Check container PID from the host:

In [ ]:
!docker inspect --format '{{.State.Pid}}' inference-api

This helps when:

```text
Application is running
        ↓
Port is active
        ↓
But ps doesn't show the expected application
        ↓
Because it is inside a container
```

---

# Phase 11: Advanced Case — Process Deleted but Still Running

A program may still be running even after its executable or file has been deleted.

Check:

In [ ]:
!sudo lsof | grep deleted

You may find:

```text
python3  12345 user  txt  /tmp/app (deleted)
```

This can cause:

```text
Disk space confusion
Memory consumption
Unexpected running services
Old application versions remaining active
```

---

# Phase 12: Resolve the Problem

Suppose you identified:

```text
PID=12345
```

## Graceful Stop

In [ ]:
!kill 12345

Wait:

In [ ]:
!sleep 3

Check:

In [ ]:
!ps -p 12345

If still running:

In [ ]:
!kill -15 12345

As a last resort:

In [ ]:
!kill -9 12345

⚠️ `kill -9` should be the last option because the application cannot clean up resources gracefully.

---

# Phase 13: Confirm Resolution

This is one of the most important parts of production troubleshooting.

Do not assume the problem is fixed.

## Confirm PID Is Gone

In [ ]:
!ps -p 12345

Expected:

```text
No process found
```

---

## Check Process Search

In [ ]:
!pgrep -af hidden_worker

Expected:

```text
No output
```

---

## Check Memory Again

In [ ]:
!free -h

Compare with baseline.

---

## Check CPU

In [ ]:
!top

or:

In [ ]:
!ps aux --sort=-%cpu | head

---

## Check Port

If this was an API:

In [ ]:
!sudo ss -ltnp | grep :8000

Expected:

```text
No process listening
```

---

# AI/ML Engineer Production Use Cases

## 1. Model Inference Server

Architecture:

```text
Client
  ↓
FastAPI
  ↓
Gunicorn
  ↓
Python Worker
  ↓
PyTorch Model
  ↓
GPU
```

Problem:

```text
API latency increases
        ↓
CPU is high
        ↓
Multiple Python workers exist
        ↓
You cannot identify which worker owns the model
```

Commands:

In [ ]:
!ps aux --sort=-%mem | head
!pstree -p
!pgrep -af python

---

## 2. GPU Memory Problem

A model process may still be running and holding GPU memory.

Check:

In [ ]:
!nvidia-smi

Example:

```text
PID      Process
12345    python
```

Then investigate:

In [ ]:
!ps -fp 12345

This is critical when:

```text
CUDA Out of Memory
        ↓
You believe training stopped
        ↓
Old Python process is still alive
        ↓
GPU memory remains occupied
```

Resolution:

In [ ]:
!kill 12345

Then confirm:

In [ ]:
!nvidia-smi

---

## 3. RAG Pipeline

Example:

```text
Document Upload
      ↓
Parsing
      ↓
Chunking
      ↓
Embedding
      ↓
Milvus
```

Problem:

```text
Embedding pipeline becomes slow
```

Investigation:

In [ ]:
!ps aux --sort=-%cpu | head

You discover:

```text
Multiple embedding workers
```

Then:

In [ ]:
!pstree -p

You identify an orphaned worker consuming CPU.

---

## 4. Training Job

Scenario:

```text
Training completed
        ↓
But CPU remains high
        ↓
GPU remains occupied
```

Check:

In [ ]:
!pgrep -af python

Then:

In [ ]:
!nvidia-smi

Possible cause:

```text
DataLoader worker
Background multiprocessing worker
Distributed training process
Zombie process
```

---

## 5. FastAPI Production Service

Problem:

```text
FastAPI is responding
```

But:

In [ ]:
!ps aux | grep fastapi

returns nothing.

Why?

Because the actual process may be:

```text
uvicorn
```

or:

```text
gunicorn
```

or:

```text
python
```

Correct investigation:

In [ ]:
!pgrep -af uvicorn
!pgrep -af gunicorn
!pgrep -af python

Also:

In [ ]:
!sudo ss -ltnp | grep :8000

---

# Final Production Mental Model

When a program is running but you cannot find its PID, think:

```text
1. What symptom do I see?
       │
       ├── High CPU?
       ├── High RAM?
       ├── GPU usage?
       ├── Open port?
       └── Logs?
             ↓
2. Search using the symptom
             ↓
3. Find PID
             ↓
4. Inspect process
             ↓
5. Inspect parent/child tree
             ↓
6. Check systemd/container
             ↓
7. Resolve
             ↓
8. Verify
             ↓
9. Measure impact
```

## Best Commands to Memorize

In [ ]:
!pgrep -af python
!ps aux --sort=-%mem | head -20
!ps aux --sort=-%cpu | head -20
!pstree -p
!sudo lsof -i :8000
!sudo ss -ltnp
!systemctl status SERVICE
!docker ps
!docker top CONTAINER
!nvidia-smi
!ps -fp PID

### Key AI/ML Engineering lesson

**Don't search only by application name. Search by the observable symptom CPU, RAM, GPU, network port, parent process, container, or systemd service and trace that evidence back to the PID.**